In [ ]:
!pip install gdown

In [ ]:
path="https://drive.google.com/file/d/1oroijAQfYHR5pZUFauhqeTFmdNrE08YB/view?usp=sharing"

In [ ]:
import gdown
file_id="1oroijAQfYHR5pZUFauhqeTFmdNrE08YB"
output_file="phishing-url.csv"
gdown.download(f"https://drive.google.com/uc?id={file_id}",output_file,fuzzy= True)



Downloading...
From: https://drive.google.com/uc?id=1oroijAQfYHR5pZUFauhqeTFmdNrE08YB
To: /content/phishing-url.csv
100%|██████████| 56.9M/56.9M [00:01<00:00, 54.1MB/s]


'phishing-url.csv'

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("/content/phishing-url.csv")

In [ ]:
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [ ]:
# remove duplicates
df= df.drop_duplicates()

In [ ]:
X=df.drop(columns=["label"])
y=df["label"]

## the dataset has no missing values and the target is balanced

let's see the distribution of the dataset features either skewed or normal distribution

## catagorical features


In [ ]:
catagorical=["FILENAME""TLD","Title","Domain","URL"]

In [ ]:

# drop `FILENAME` feature
X = X.drop(columns=["FILENAME"])

## TLD(top levle domain)
TLD stands for Top‑Level Domain, i.e., the last part of the domain after the last dot:
com, org, net, edu, uk, br, ru, zw, etc.

So when your feature TLD has 695 unique values, it means:

  * Your dataset contains URLs spread across 695 different TLDs (mostly .com, .org, .net, plus many country‑code TLDs like .uk, .de, .fr, .jp, etc.).

What this tells you

* Domain diversity

  * The dataset is very diverse geographically and by domain type (generic .com, sponsored TLDs, country codes).

  * This is good for generalization: your model will see unusual TLDs that attackers sometimes abuse.

* Modeling challenge

  * TLD is categorical with high cardinality (695 categories).

  * If you do one‑hot encoding, you get 695 new binary columns, which can:

    * Inflate memory.

            Create many sparse / rarely‑used features (if 680 TLDs appear only a few times).

* Security / phishing insight

        Research shows that less common or new TLDs are sometimes associated with higher phishing risk because they look “cheap” or “suspicious” to users.

## instead of encoding 695 different values we select the top 30 values and other create other feature

In [ ]:
N = 30   # choose how many top TLDs to keep (e.g., 20–50)

# Get top N TLDs by frequency
top_tlds = X["TLD"].value_counts().head(N).index

# Create new reduced TLD feature
X["TLD_reduced"] = X["TLD"].where(X["TLD"].isin(top_tlds), "other_tld")

# Optional: convert to categorical to make one‑hot safer
X["TLD_reduced"] = X["TLD_reduced"].astype("category")

In [ ]:
X = X.drop(columns=["TLD"])   # drop the original high‑cardinality TLD

**Title**

**since `Title` feature contains around 197874 different values(catagorical) encoding this different values break the model almost many of them are unique it does not give any pattern for our model.so we drop `Title` feature**

In [ ]:
X = X.drop(columns=["Title"])

**Domain**

**`Domain` feature we already extract necessary information from the feature we only extract one feature called `Domain entropy` that tells the randomness of the doamin malicius domains are their entropy is high which is a stong signal for our model,then we can drop the feature**

In [ ]:
import math
from collections import Counter

def domain_entropy(domain):
    if pd.isna(domain) or len(domain) == 0:
        return 0.0
    chars = [c.lower() for c in domain if c.isalnum()]
    if len(chars) == 0:
        return 0.0
    freq = Counter(chars)
    probs = [freq[c] / len(chars) for c in freq]
    return -sum(p * math.log2(p) for p in probs)

In [ ]:
X["DomainEntropy"] = X["Domain"].apply(domain_entropy)
X = X.drop(columns=["Domain"])   # now drop the raw high‑cardinality column

**url**

**almost many of them neccessary features are extract from url features but before i drop the feature i add the following new features:**
- has_ip_in_url
- is_shortened
- path_depth
- has_port
- has_login_keyword
- has_bank_or_pay_keyword
- url_entropy

In [ ]:
from urllib.parse import urlparse
import re

def extract_url_extra_features(url):
    if not isinstance(url, str) or not url.strip():
        return {
            "has_ip_in_url": 0,
            "is_shortened": 0,
            "path_depth": 0,
            "has_port": 0,
            "has_login_keyword": 0,
            "has_bank_or_pay_keyword": 0,
            "url_entropy": 0.0,
        }

    # Parse URL
    parsed = urlparse(url.lower())
    domain = parsed.netloc
    path = parsed.path

    # 1. has_ip_in_url
    has_ip_in_url = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0

    # 2. is_shortened
    shorteners = ["bit.ly", "tinyurl", "goo.gl", "t.co", "cutt.ly", "adf.ly", "ow.ly"]
    is_shortened = 1 if any(tok in url for tok in shorteners) else 0

    # 3. path_depth
    path_parts = [p for p in path.strip("/").split("/") if p]
    path_depth = len(path_parts)

    # 4. has_port
    has_port = 1 if ":" in domain and "@" not in domain else 0

    # 5. has_login_keyword
    login_words = ["login", "signin", "sign-in", "log-in", "auth", "authenticate", "signin"]
    has_login_keyword = 1 if any(w in url for w in login_words) else 0

    # 6. has_bank_or_pay_keyword
    bank_pay_words = [
        "bank", "paypal", "payment", "pay", "bill", "invoice",
        "verify", "verification", "account", "secure", "update",
    ]
    has_bank_or_pay_keyword = 1 if any(w in url for w in bank_pay_words) else 0

    # 7. url_entropy (Shannon entropy)
    chars = [c for c in url if c.isalnum()]
    if len(chars) == 0:
        url_entropy = 0.0
    else:
        from collections import Counter
        import math
        freq = Counter(chars)
        probs = [freq[c] / len(chars) for c in freq]
        url_entropy = -sum(p * math.log2(p) for p in probs)

    return {
        "has_ip_in_url": has_ip_in_url,
        "is_shortened": is_shortened,
        "path_depth": path_depth,
        "has_port": has_port,
        "has_login_keyword": has_login_keyword,
        "has_bank_or_pay_keyword": has_bank_or_pay_keyword,
        "url_entropy": url_entropy,
    }

In [ ]:
X.columns

Index(['URL', 'URLLength', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex',
       'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength',
       'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar',
       'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
       'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL',
       'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength',
       'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon',
       'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect',
       'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit',
       'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields',
       'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo',
       'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef',
       'NoOfExternalRef', 'TLD_reduced', 'DomainEntropy']

In [ ]:
# Apply to your url column
extra_feats = X["URL"].apply(extract_url_extra_features)

# Convert to DataFrame and join
X = pd.concat([X, pd.DataFrame(list(extra_feats))], axis=1)

# Now safely drop the raw high-cardinality url
X = X.drop(columns=["URL"])

## converting catagorical features into numeric(Encoding )

In [ ]:


from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, drop=None)
# Reshape for single column
encoded = encoder.fit_transform(X[["TLD_reduced"]])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())

# Combine with original dataset (drop original column)
df_encoded = pd.concat([X.drop(columns=["TLD_reduced"]), encoded_df], axis=1)



In [ ]:
df_encoded.shape

(235795, 89)

## modeling
we will try the following classification models
1. Start with tree‑based models (best “default” choice)

    Random Forest

        Very robust to:

            Mixed numeric/categorical features.

            Slightly noisy or redundant features (e.g., from TLD, Domain, Title cleanup).

        Often achieves >97% accuracy on phishing‑URL datasets in the literature.

    XGBoost / LightGBM / CatBoost

        Usually slightly better than Random Forest in accuracy and speed, and good at handling imbalance and feature importance.

        Works well when you have many numeric features (URL lengths, ratios, obfuscation stats, etc.).
2. If you want a simpler “baseline”

    Logistic Regression

        Interpretable baseline; good for:

            Checking if your feature‑engineering is meaningful (if LR underperforms badly, you may need to fix features).

        Usually slightly worse than Random Forest on this kind of task

    naive baye's
    svm

## drop the features that extract from fetching from html

In [ ]:
dropped_features = [
    'LineOfCode',
    'LargestLineLength',
    'HasTitle',
    'DomainTitleMatchScore',
    'URLTitleMatchScore',
    'HasDescription',

    'NoOfImage',
    'NoOfCSS',
    'NoOfJS',

    'NoOfSelfRef',
    'NoOfEmptyRef',
    'NoOfExternalRef',

    'NoOfURLRedirect',
    'NoOfSelfRedirect',

    'HasFavicon',
    'HasSubmitButton',
    'HasHiddenFields',
    'HasPasswordField',
    'HasExternalFormSubmit',

    'NoOfPopup',
    'NoOfiFrame',

    'Robots',
    'IsResponsive',

    'HasSocialNet',
    'HasCopyrightInfo',

]


In [ ]:
df_encoded = df_encoded.drop(columns=dropped_features)

## select features instead of bulding the whole feature on the development

In [ ]:
# Calculate Mutual Information
from sklearn.feature_selection import mutual_info_classif, SelectKBest
mi_scores = mutual_info_classif(df_encoded, y, random_state=42)
mi_scores_series = pd.Series(mi_scores, index=df_encoded.columns)
mi_scores_series = mi_scores_series.sort_values(ascending=False)

print("🔥 Top 15 MI Features:")
print(mi_scores_series.head(30))

🔥 Top 15 MI Features:
URLSimilarityIndex            0.680838
LetterRatioInURL              0.382441
IsHTTPS                       0.256511
NoOfOtherSpecialCharsInURL    0.241039
SpacialCharRatioInURL         0.205411
TLDLegitimateProb             0.200665
url_entropy                   0.173279
CharContinuationRate          0.172766
DegitRatioInURL               0.170930
NoOfDegitsInURL               0.170925
URLLength                     0.167768
NoOfLettersInURL              0.152870
DomainEntropy                 0.148572
URLCharProb                   0.137236
path_depth                    0.109093
DomainLength                  0.088862
NoOfSubDomain                 0.083667
Pay                           0.081163
TLDLength                     0.031778
TLD_reduced_app               0.022995
NoOfQMarkInURL                0.022810
Bank                          0.021977
TLD_reduced_com               0.021556
NoOfEqualsInURL               0.019802
TLD_reduced_org               0.018305
has

In [ ]:
selected_features = [
    "URLSimilarityIndex",
    "LetterRatioInURL",
    "IsHTTPS",
    "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL",
    "TLDLegitimateProb",
    "url_entropy",
    "CharContinuationRate",
    "DegitRatioInURL",
    "NoOfDegitsInURL",
    "URLLength",
    "NoOfLettersInURL",
    "DomainEntropy",
    "URLCharProb",
    "path_depth",
    "DomainLength",
    "NoOfSubDomain",
    "Pay",
    "TLDLength",
    "TLD_reduced_app",
    "NoOfQMarkInURL",
    "Bank",
    "TLD_reduced_com",
    "NoOfEqualsInURL",
    "TLD_reduced_org",
    "has_login_keyword",
    "TLD_reduced_co",
    "has_bank_or_pay_keyword",
    "TLD_reduced_io",
    "TLD_reduced_uk"
]

# Extract ONLY these 22 features
X_selected = df_encoded[selected_features].copy()

# Verify all exist
missing_features = [f for f in selected_features if f not in df_encoded.columns]
if missing_features:
    print("⚠️  Missing features:", missing_features)
else:
    print(f"✅ All 22 features found! Shape: {X_selected.shape}")
    print("Features:")
    for i, feature in enumerate(selected_features, 1):
        print(f"{i:2d}. {feature}")

✅ All 22 features found! Shape: (235795, 30)
Features:
 1. URLSimilarityIndex
 2. LetterRatioInURL
 3. IsHTTPS
 4. NoOfOtherSpecialCharsInURL
 5. SpacialCharRatioInURL
 6. TLDLegitimateProb
 7. url_entropy
 8. CharContinuationRate
 9. DegitRatioInURL
10. NoOfDegitsInURL
11. URLLength
12. NoOfLettersInURL
13. DomainEntropy
14. URLCharProb
15. path_depth
16. DomainLength
17. NoOfSubDomain
18. Pay
19. TLDLength
20. TLD_reduced_app
21. NoOfQMarkInURL
22. Bank
23. TLD_reduced_com
24. NoOfEqualsInURL
25. TLD_reduced_org
26. has_login_keyword
27. TLD_reduced_co
28. has_bank_or_pay_keyword
29. TLD_reduced_io
30. TLD_reduced_uk


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df_encoded, y, test_size=0.2, random_state=42)

### models and their accuracy score
1. RandomforestClassifier   ---99
2. linearRegressio           --99
3. xgboost         ---100%

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,           # L1 regularization (alpha)
    reg_lambda=1.0,          # L2 regularization (lambda)
    random_state=42,
    objective='binary:logistic'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# import classification report
from sklearn.metrics import classification_report
y_pred_v3 = model.predict(X_test)

print("=== After proper split ===")
print(classification_report(y_test, y_pred_v3))

=== After proper split ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20124
           1       1.00      1.00      1.00     27035

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



In [ ]:
import joblib

# Save trained model
joblib.dump(model, "phishing_url_detector_xgb.pkl")

# Later, load it:
# model_xgb = joblib.load("phishing_url_detector_xgb.pkl")

## Refactoring Preprocessing into Functions

To make the preprocessing pipeline more structured and suitable for deployment, we will consolidate the steps into a series of functions.

First, let's define the helper functions for `domain_entropy` and `extract_url_extra_features`.

In [ ]:
import math
from collections import Counter
from urllib.parse import urlparse
import re
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

def calculate_domain_entropy(domain):
    """Calculates the Shannon entropy of a domain string."""
    if pd.isna(domain) or len(domain) == 0:
        return 0.0
    chars = [c.lower() for c in domain if c.isalnum()]
    if len(chars) == 0:
        return 0.0
    freq = Counter(chars)
    probs = [freq[c] / len(chars) for c in freq]
    return -sum(p * math.log2(p) for p in probs)

def extract_url_extra_features(url):
    """Extracts several extra features from a URL string."""
    if not isinstance(url, str) or not url.strip():
        return {
            "has_ip_in_url": 0,
            "is_shortened": 0,
            "path_depth": 0,
            "has_port": 0,
            "has_login_keyword": 0,
            "has_bank_or_pay_keyword": 0,
            "url_entropy": 0.0,
        }

    parsed = urlparse(url.lower())
    domain = parsed.netloc
    path = parsed.path

    has_ip_in_url = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0
    shorteners = ["bit.ly", "tinyurl", "goo.gl", "t.co", "cutt.ly", "adf.ly", "ow.ly"]
    is_shortened = 1 if any(tok in url for tok in shorteners) else 0
    path_parts = [p for p in path.strip("/").split("/") if p]
    path_depth = len(path_parts)
    has_port = 1 if ":" in domain and "@" not in domain else 0
    login_words = ["login", "signin", "sign-in", "log-in", "auth", "authenticate", "signin"]
    has_login_keyword = 1 if any(w in url for w in login_words) else 0
    bank_pay_words = [
        "bank", "paypal", "payment", "pay", "bill", "invoice",
        "verify", "verification", "account", "secure", "update",
    ]
    has_bank_or_pay_keyword = 1 if any(w in url for w in bank_pay_words) else 0

    chars = [c for c in url if c.isalnum()]
    if len(chars) == 0:
        url_entropy = 0.0
    else:
        freq = Counter(chars)
        probs = [freq[c] / len(chars) for c in freq]
        url_entropy = -sum(p * math.log2(p) for p in probs)

    return {
        "has_ip_in_url": has_ip_in_url,
        "is_shortened": is_shortened,
        "path_depth": path_depth,
        "has_port": has_port,
        "has_login_keyword": has_login_keyword,
        "has_bank_or_pay_keyword": has_bank_or_pay_keyword,
        "url_entropy": url_entropy,
    }


Next, we'll create a comprehensive `preprocess_pipeline` function that takes raw features, processes them, and returns the cleaned, engineered features. This function will also manage the `OneHotEncoder` and the list of top TLDs for consistent processing of new data.

In [ ]:
def preprocess_pipeline(X_raw, encoder=None, top_tlds_list=None, n_top_tlds=30, is_train=True):
    """Applies all preprocessing steps to the input DataFrame X_raw.

    Args:
        X_raw (pd.DataFrame): The raw feature DataFrame.
        encoder (OneHotEncoder, optional): Fitted OneHotEncoder for TLD_reduced.
                                           If None and is_train=True, a new encoder is fitted.
        top_tlds_list (list, optional): List of top TLDs to consider.
                                        If None and is_train=True, it's calculated.
        n_top_tlds (int): Number of top TLDs to keep if calculating from scratch.
        is_train (bool): Flag indicating if the data is for training (True) or new data (False).

    Returns:
        tuple: (X_processed, encoder, top_tlds_list).
               encoder and top_tlds_list are returned only if is_train is True
               or if they were passed in.
    """
    X_processed = X_raw.copy()

    # Drop FILENAME and Title
    if "FILENAME" in X_processed.columns:
        X_processed = X_processed.drop(columns=["FILENAME"])
    if "Title" in X_processed.columns:
        X_processed = X_processed.drop(columns=["Title"])

    # Process Domain
    if "Domain" in X_processed.columns:
        X_processed["DomainEntropy"] = X_processed["Domain"].apply(calculate_domain_entropy)
        X_processed = X_processed.drop(columns=["Domain"])

    # Process URL
    if "URL" in X_processed.columns:
        extra_feats = X_processed["URL"].apply(extract_url_extra_features)
        # Ensure the DataFrame created from extra_feats has the same index as X_processed
        extra_feats_df = pd.DataFrame(list(extra_feats), index=X_processed.index)
        X_processed = pd.concat([X_processed, extra_feats_df], axis=1)
        X_processed = X_processed.drop(columns=["URL"])

    # Process TLD
    if "TLD" in X_processed.columns:
        if is_train and top_tlds_list is None:
            # Calculate top TLDs from training data
            top_tlds_list = X_processed["TLD"].value_counts().head(n_top_tlds).index.tolist()

        if top_tlds_list is not None:
            X_processed["TLD_reduced"] = X_processed["TLD"].where(X_processed["TLD"].isin(top_tlds_list), "other_tld")
            X_processed["TLD_reduced"] = X_processed["TLD_reduced"].astype("category")
        else:
            # Fallback if not training and no top_tlds_list provided
            X_processed["TLD_reduced"] = "other_tld"
            X_processed["TLD_reduced"] = X_processed["TLD_reduced"].astype("category")

        X_processed = X_processed.drop(columns=["TLD"])

        # One-Hot Encode TLD_reduced
        if is_train and encoder is None:
            encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # handle_unknown for deployment
            encoded_tld = encoder.fit_transform(X_processed[["TLD_reduced"]])
        elif encoder is not None:
            encoded_tld = encoder.transform(X_processed[["TLD_reduced"]])
        else:
            raise ValueError("Encoder must be provided for non-training data.")

        encoded_tld_df = pd.DataFrame(encoded_tld, columns=encoder.get_feature_names_out(['TLD_reduced']),
                                      index=X_processed.index)
        X_processed = pd.concat([X_processed.drop(columns=["TLD_reduced"]), encoded_tld_df], axis=1)

    if is_train:
        return X_processed, encoder, top_tlds_list
    else:
        return X_processed

Now, let's apply the `preprocess_pipeline` to the entire dataset to prepare `X` and `y` before splitting into training and test sets. We'll capture the `encoder` and `top_tlds_list` from this step.

In [ ]:
# Assuming 'df' is your initial DataFrame loaded from '/content/phishing-url.csv'
# and duplicates have been dropped as per previous steps.
# If starting from scratch, include loading and dropping duplicates here.

# Re-load df and drop duplicates to ensure a clean start for the pipeline
initial_df = pd.read_csv("/content/phishing-url.csv")
initial_df = initial_df.drop_duplicates()

# Separate features (X_full_raw) and target (y)
X_full_raw = initial_df.drop(columns=["label"])
y_full = initial_df["label"]

# Apply preprocessing pipeline to the full dataset (acting as initial training)
X_processed_full, tld_encoder, top_tlds_for_deployment = preprocess_pipeline(
    X_full_raw,
    is_train=True
)

print("Shape of processed full dataset:", X_processed_full.shape)
print("Columns after preprocessing:", X_processed_full.columns.tolist())


Shape of processed full dataset: (235795, 89)
Columns after preprocessing: ['URLLength', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'DomainEntropy', 'has_ip_in_url', 'is_shortened', 'path_dept

With the preprocessing pipeline defined, we can now perform the train-test split using the processed full dataset. Then, we will ensure `X_test` is processed consistently using the `tld_encoder` and `top_tlds_for_deployment` captured from the training set preprocessing.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the processed full dataset
X_train_processed, X_test_raw, y_train, y_test = train_test_split(
    X_full_raw, y_full, test_size=0.2, random_state=42, stratify=y_full # Added stratify for balanced split
)

# Preprocess X_train separately to fit the encoder and get top TLDs (if not done on full dataset)
# In a real scenario, you'd apply the pipeline to X_train first to get encoder and top_tlds_list
# Since we already ran it on the full dataset, we'll use the results from X_processed_full.
# For proper deployment, X_train should ideally be passed through preprocess_pipeline to fit,
# and then X_test passed through to transform.

# To simulate a proper train/test split preprocessing:
# We need to re-run the pipeline on X_train to *fit* the encoder and identify top TLDs
# and then use those fitted objects to *transform* X_test.

# First, get processed X_train and the fitted encoder/top_tlds_list from X_train itself
X_train_final, fitted_encoder, trained_top_tlds = preprocess_pipeline(
    X_train_processed,
    is_train=True,
    n_top_tlds=30
)

# Then, use the fitted_encoder and trained_top_tlds to transform X_test_raw
X_test_final = preprocess_pipeline(
    X_test_raw,
    encoder=fitted_encoder,
    top_tlds_list=trained_top_tlds,
    is_train=False
)

print("Shape of X_train_final:", X_train_final.shape)
print("Shape of X_test_final:", X_test_final.shape)


Shape of X_train_final: (188636, 89)
Shape of X_test_final: (47159, 89)


You now have `X_train_final`, `y_train`, `X_test_final`, and `y_test` ready for model training. The `fitted_encoder` and `trained_top_tlds` objects are essential for consistently preprocessing new, unseen data with the trained model during deployment.

In [ ]:
from xgboost import XGBClassifier

model2 = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,           # L1 regularization (alpha)
    reg_lambda=1.0,          # L2 regularization (lambda)
    random_state=42,
    objective='binary:logistic'
)

model2.fit(X_train_final, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# import classification report
from sklearn.metrics import classification_report
y_pred_v3 = model2.predict(X_test_final)

print("=== After proper split ===")
print(classification_report(y_test, y_pred_v3))

=== After proper split ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159

